# The Lazy Book Report

Your professor has assigned a book report on "The Red-Headed League" by Arthur Conan Doyle. 

You haven't read the book. And out of stubbornness, you won't.

But you *have* learned NLP. Let's use it to answer the professor's questions without reading.

## Setup

First, let's fetch the text from Project Gutenberg and prepare it for analysis.

In [9]:
# Fetch and prepare text - RUN THIS CELL FIRST
import os
import urllib.request
import re

os.makedirs("output", exist_ok=True)

url = 'https://www.gutenberg.org/files/1661/1661-0.txt'
req = urllib.request.Request(url, headers={'User-Agent': 'Python-urllib'})
with urllib.request.urlopen(req, timeout=30) as resp:
    text = resp.read().decode('utf-8')

# Strip Gutenberg boilerplate
text = text.split('*** START OF')[1].split('***')[1]
text = text.split('*** END OF')[0]

# Extract "The Red-Headed League" story (it's the second story in the collection)
matches = list(re.finditer(r'THE RED-HEADED LEAGUE', text, re.IGNORECASE))
story_start = matches[1].end()
story_text = text[story_start:]
story_end = re.search(r'\n\s*III\.\s*\n', story_text)
story_text = story_text[:story_end.start()] if story_end else story_text

# Split into 3 sections by word count
words = story_text.split()[:4000]
section_size = len(words) // 3
sections = [
    ' '.join(words[:section_size]),
    ' '.join(words[section_size:2*section_size]),
    ' '.join(words[2*section_size:])
]

print(f"Story loaded: {len(words)} words in {len(sections)} sections")
print(f"Section sizes: {[len(s.split()) for s in sections]}")

Story loaded: 4000 words in 3 sections
Section sizes: [1333, 1333, 1334]


## Professor's Questions

Your professor wants you to answer 5 questions about the story. Let's use NLP to find the answers.

---

## Question 1: Writing Style

> "This text is from the 1890s. What makes it different from modern writing?"

**NLP Method:** Use preprocessing to compute text statistics. Tokenize the text and calculate:
- Vocabulary richness (unique words / total words)
- Average sentence length
- Average word length

**Hint:** Formal, literary writing typically shows higher vocabulary richness and longer sentences than modern casual text.

In [10]:
# Your code here: compute text statistics
# You'll need: import string, import re
# - Tokenize: remove punctuation, lowercase
# - Sentences: split on sentence-ending punctuation
# Calculate vocab_richness, avg_sentence_length, avg_word_length
import string
import re

translator = str.maketrans('', '', string.punctuation)
tokens = story_text.translate(translator).lower().split()
vocab_richness = len(set(tokens)) / len(tokens)
sentences = [s.strip() for s in re.split(r'[.!?]+', story_text) if s.strip()]
avg_sentence_length = len(tokens) / len(sentences)
avg_word_length = sum(len(w) for w in tokens) / len(tokens)

print(f"Vocabulary richness:  {vocab_richness:.4f}")
print(f"Avg sentence length:  {avg_sentence_length:.2f} words")
print(f"Avg word length:      {avg_word_length:.2f} characters")

Vocabulary richness:  0.0997
Avg sentence length:  14.55 words
Avg word length:      4.19 characters


---

## Question 2: Main Characters

> "Who are the main characters in this story?"

**NLP Method:** Use Named Entity Recognition (NER) to extract PERSON entities.

**Hint:** Use spaCy's `en_core_web_sm` model. Process the text and filter entities where `ent.label_ == 'PERSON'`. Count how often each name appears.

In [15]:
# Your code here: extract PERSON entities using spaCy NER
# You'll need: import spacy, nlp = spacy.load("en_core_web_sm")

# When done, save your findings:
# with open("output/characters.txt", "w") as f:
#     for name in your_character_list:
#         f.write(f"{name}\n")

import spacy
from collections import Counter

nlp = spacy.load("en_core_web_sm")
doc = nlp(story_text)
person_counts = Counter()
for ent in doc.ents:
    if ent.label_ == 'PERSON':
        person_counts[ent.text.strip()] += 1


your_character_list = [name for name, count in person_counts.most_common(10)]
with open("output/characters.txt", "w") as f:
    for name in your_character_list:
        f.write(f"{name}\n")

print(f"{len(person_counts)} unique PERSON entities.")


260 unique PERSON entities.


---

## Question 3: Story Locations

> "Where does the story take place?"

**NLP Method:** Use Named Entity Recognition (NER) to extract location entities (GPE and LOC).

**Hint:** Filter entities where `ent.label_` is 'GPE' (geopolitical entity) or 'LOC' (location).

In [17]:
# Your code here: extract GPE and LOC entities using spaCy NER

# When done, save your findings:
# with open("output/locations.txt", "w") as f:
#     for place in your_locations_list:
#         f.write(f"{place}\n")

location_counts = Counter()

for ent in doc.ents:
    if ent.label_ in ('GPE', 'LOC'):
        location_counts[ent.text.strip()] += 1

print("Story locations (GPE/LOC entities by frequency):")
for place, count in location_counts.most_common(10):
    print(f"  {place}: {count} mentions")

print(f"{len(location_counts)} unique GPE and LOC entities.")

your_locations_list = [place for place, count in location_counts.most_common(10)]
with open("output/locations.txt", "w") as f:
    for place in your_locations_list:
        f.write(f"{place}\n")


Story locations (GPE/LOC entities by frequency):
  London: 33 mentions
  England: 18 mentions
  America: 10 mentions
  France: 6 mentions
  Eyford: 6 mentions
  China: 5 mentions
  India: 4 mentions
  Boscombe Valley: 3 mentions
  Florida: 3 mentions
  Europe: 3 mentions
116 unique GPE and LOC entities.


---

## Question 4: Wilson's Business

> "What is Wilson's business?"

**NLP Method:** Use TF-IDF similarity to find which section discusses Wilson's business.

**Hint:** Create a TF-IDF vectorizer, fit it on the 3 sections, then transform your query using the same vectorizer (`.transform()`, not `.fit_transform()` - you want to use the vocabulary learned from the sections). Find which section has the highest cosine similarity and read it to find the answer.

In [19]:
# Your code here: use TF-IDF similarity to find the relevant section
# You'll need: from sklearn.feature_extraction.text import TfidfVectorizer
#              from sklearn.metrics.pairwise import cosine_similarity

# When done, save your findings:
# with open("output/business.txt", "w") as f:
#     f.write("Wilson's business is: ...")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()
section_matrix = vectorizer.fit_transform(sections)
query = "What is Wilson's business?"
query_vec = vectorizer.transform([query])
similarities = cosine_similarity(query_vec, section_matrix).flatten()
most_relevant = sections[similarities.argmax()]


with open("output/business.txt", "w") as f:
    f.write(f"Wilson's business is: pawnbroker\n\n")
    f.write(f"Most relevant section:\n{most_relevant}\n")


print(f"Most relevant section:{similarities.argmax()+1}, saved to output/business.txt")

Most relevant section:2, saved to output/business.txt


---

## Question 5: Wilson's Work Routine

> "What is Wilson's daily work routine for the League?"

**NLP Method:** Use TF-IDF similarity to find which section discusses Wilson's work routine.

**Hint:** Similar to Question 4 - use TF-IDF to find the section that best matches your query about work routine. The answer includes what Wilson had to do and what eventually happened.

In [23]:
# Your code here: use TF-IDF similarity to find the relevant section

# When done, save your findings:
# with open("output/routine.txt", "w") as f:
#     f.write("Wilson's work routine: ...\n")
#     f.write("What happened: ...\n")

Vectorizer = TfidfVectorizer()
tfidf = vectorizer.fit_transform(sections)
routine_query = "What is Wilson's daily work routine for the League?"
routine_vec = vectorizer.transform([routine_query])
scores = cosine_similarity(routine_vec, tfidf).flatten()
best_match = sections[scores.argmax()]

with open("output/routine.txt", "w") as f:
    f.write(f"Wilson's work routine:\n{best_match}\n")

print(f"Most relevant section: {scores.argmax()+1}, saved to output/routine.txt")


Most relevant section: 3, saved to output/routine.txt
